# QKSR / RSIAT on Google Colab

Notebook chạy CIFAR-100 ở độ phân giải 224 với hai thí nghiệm chính:

- `adapter_cifar224.json`: RSIAT + QKSR.
- `adapter_cifar224_baseline.json`: RSIAT gốc, tắt QKSR.

Mã nguồn chạy trên SSD tạm của Colab; dataset, log và checkpoint được lưu trên Google Drive. Chọn `upload` để tải project dạng ZIP từ máy, hoặc `git` để clone repository.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import os, shutil, subprocess, zipfile

SOURCE_MODE = 'upload'  # @param ["upload", "git"]
REPO_URL = 'https://github.com/PhThuan-tech/RSIAT.git'  # @param {type:'string'}
BRANCH = 'main'  # @param {type:'string'}
PROJECT_DIR = Path('/content/QG-RSA')
OUTPUT_ROOT = Path('/content/drive/MyDrive/QKSR_runs')
PERSISTENT_DATA_DIR = Path('/content/drive/MyDrive/QKSR_data/datasets')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
PERSISTENT_DATA_DIR.mkdir(parents=True, exist_ok=True)

def safe_extract_zip(archive_path, destination):
    destination = destination.resolve()
    with zipfile.ZipFile(archive_path) as archive:
        for member in archive.infolist():
            member_path = (destination / member.filename).resolve()
            if member_path != destination and destination not in member_path.parents:
                raise ValueError(f'Unsafe path in ZIP: {member.filename}')
        archive.extractall(destination)

if SOURCE_MODE == 'upload':
    from google.colab import files

    print('Chọn một file .zip chứa toàn bộ project QG-RSA.')
    uploaded = files.upload()
    zip_files = [(name, content) for name, content in uploaded.items() if name.lower().endswith('.zip')]
    if len(zip_files) != 1:
        raise ValueError('Hãy upload đúng một file .zip.')

    upload_name, upload_content = zip_files[0]
    archive_path = Path('/content') / Path(upload_name).name
    archive_path.write_bytes(upload_content)
    extract_dir = Path('/content/qksr_uploaded_source')
    if extract_dir.exists():
        shutil.rmtree(extract_dir)
    extract_dir.mkdir(parents=True)
    safe_extract_zip(archive_path, extract_dir)

    candidates = sorted({
        path.parent for path in extract_dir.rglob('main.py')
        if (path.parent / 'exps').is_dir()
    })
    if len(candidates) != 1:
        raise RuntimeError(
            f'Không xác định được duy nhất project root (tìm thấy {len(candidates)}). '
            'ZIP cần chứa main.py và thư mục exps/ trong cùng một thư mục.'
        )
    if PROJECT_DIR.exists():
        shutil.rmtree(PROJECT_DIR)
    shutil.copytree(candidates[0], PROJECT_DIR)
elif SOURCE_MODE == 'git':
    if (PROJECT_DIR / '.git').exists():
        subprocess.run(
            ['git', '-C', str(PROJECT_DIR), 'pull', '--ff-only', 'origin', BRANCH],
            check=True,
        )
    else:
        if PROJECT_DIR.exists():
            shutil.rmtree(PROJECT_DIR)
        subprocess.run(
            ['git', 'clone', '--branch', BRANCH, REPO_URL, str(PROJECT_DIR)],
            check=True,
        )
else:
    raise ValueError("SOURCE_MODE phải là 'upload' hoặc 'git'.")

required_paths = [
    PROJECT_DIR / 'main.py',
    PROJECT_DIR / 'exps' / 'adapter_cifar224.json',
    PROJECT_DIR / 'exps' / 'adapter_cifar224_baseline.json',
]
missing = [str(path) for path in required_paths if not path.exists()]
if missing:
    raise FileNotFoundError('Project thiếu file bắt buộc: ' + ', '.join(missing))

local_data_dir = PROJECT_DIR / 'data' / 'datasets'
if local_data_dir.is_symlink():
    if local_data_dir.resolve() != PERSISTENT_DATA_DIR.resolve():
        local_data_dir.unlink()
elif local_data_dir.exists():
    if any(local_data_dir.iterdir()):
        shutil.copytree(local_data_dir, PERSISTENT_DATA_DIR, dirs_exist_ok=True)
    shutil.rmtree(local_data_dir)
local_data_dir.parent.mkdir(parents=True, exist_ok=True)
if not local_data_dir.exists():
    local_data_dir.symlink_to(PERSISTENT_DATA_DIR, target_is_directory=True)

os.chdir(PROJECT_DIR)
print('Project:', Path.cwd())
print('Dataset:', PERSISTENT_DATA_DIR)
print('Outputs:', OUTPUT_ROOT)

In [ ]:
!nvidia-smi
import torch, torchvision
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')
print('torchvision:', torchvision.__version__)

# Giữ torch/torchvision có sẵn trên Colab, chỉ cài dependency của project.
%pip install -q --upgrade-strategy only-if-needed -r requirements-colab.txt
import timm
print('timm:', timm.__version__)
!python -m py_compile main.py trainer.py models/base.py models/RSIAT_adapter.py utils/quantum_kernel.py utils/qksr_statistics.py

In [ ]:
# Tải CIFAR-100 một lần và giữ dataset trên Google Drive.
from torchvision.datasets import CIFAR100
import json

try:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=False)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=False)
    print('Đã tìm thấy CIFAR-100 trên Drive; bỏ qua download.')
except RuntimeError:
    cifar_train = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=True, download=True)
    cifar_test = CIFAR100(root=str(PERSISTENT_DATA_DIR), train=False, download=True)

def make_colab_config(source, destination, prefix, seed=1993, resume=True):
    config = json.loads(Path(source).read_text(encoding='utf-8'))
    config.update({
        'prefix': prefix,
        'seed': [seed],
        'resume': resume,
        'output_root': str(OUTPUT_ROOT),
        'device': ['0'],
        'num_workers': 2,
        'stats_num_workers': 2,
    })
    Path(destination).write_text(
        json.dumps(config, indent=2, ensure_ascii=False),
        encoding='utf-8',
    )

QKSR_CONFIG = Path('/content/qksr_cifar224.json')
BASELINE_CONFIG = Path('/content/baseline_cifar224.json')
make_colab_config('exps/adapter_cifar224.json', QKSR_CONFIG, prefix='qksr')
make_colab_config('exps/adapter_cifar224_baseline.json', BASELINE_CONFIG, prefix='baseline')

print('CIFAR train/test:', len(cifar_train), len(cifar_test))
print('QKSR config:', QKSR_CONFIG)
print('Baseline config:', BASELINE_CONFIG)

## Kiểm tra nhanh (khuyến nghị)

Chạy unit test của quantum kernel trước khi bắt đầu thí nghiệm dài. Đây không phải một cấu hình thí nghiệm CIFAR riêng.

In [ ]:
!python -m pytest -q tests/test_quantum_kernel.py

## Thí nghiệm 1 — CIFAR224 + QKSR

Checkpoint được lưu với prefix `qksr`. `resume=True` nên có thể chạy lại cell để tiếp tục task gần nhất. Bỏ dấu `#` để chạy.

In [ ]:
# !python main.py --config /content/qksr_cifar224.json

## Thí nghiệm 2 — CIFAR224 baseline (RSIAT gốc)

Cấu hình này giữ cùng hyperparameter nhưng tắt QKSR, dùng làm đối chứng trực tiếp. Checkpoint được tách bằng prefix `baseline`. Bỏ dấu `#` để chạy.

In [ ]:
# !python main.py --config /content/baseline_cifar224.json

## Kiểm tra output gần nhất

In [ ]:
print('Recent logs:')
for path in sorted((OUTPUT_ROOT / 'logs' / 'adapter').rglob('*.log'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)
print('Recent checkpoints:')
for path in sorted((OUTPUT_ROOT / 'ckpt').rglob('task_*.pkl'), key=lambda p: p.stat().st_mtime)[-10:]:
    print(path)